In [1]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

### Step 1: Preprocessing & Exploratory Data Analysis (EDA)
We will now clean the dataset, analyze length distributions, identify potential label leakage (explicit ratings), and create our stratified splits.

In [5]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Convert HF datasets to Pandas DataFrames for easier manipulation
train_df = pd.DataFrame(ds['train'])
test_df = pd.DataFrame(ds['test'])

print(f"Original train size: {len(train_df)}")
print(f"Original test size: {len(test_df)}")

Original train size: 25000
Original test size: 25000


In [7]:
train_df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [8]:
def clean_html(text):
    # Strip HTML tags like <br />
    return re.sub(r'<[^>]*>', ' ', text)

def find_explicit_ratings(text):
    # Match patterns like 8/10, 8 out of 10, 8/10., etc.
    pattern = r'\b\d{1,2}\s*(?:/|out of)\s*10\b'
    return len(re.findall(pattern, text, re.IGNORECASE)) > 0

# 1. HTML Cleaning
train_df['cleaned_text'] = train_df['text'].apply(clean_html)
test_df['cleaned_text'] = test_df['text'].apply(clean_html)

In [9]:
train_df.head()

,text,label,cleaned_text
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,I rented I AM CURIOUS-YELLOW from my video sto...
1,"""I Am Curious: Yellow"" is a risible and preten...",0,"""I Am Curious: Yellow"" is a risible and preten..."
2,If only to avoid making this type of film in t...,0,If only to avoid making this type of film in t...
3,This film was probably inspired by Godard's Ma...,0,This film was probably inspired by Godard's Ma...
4,"Oh, brother...after hearing about this ridicul...",0,"Oh, brother...after hearing about this ridicul..."


In [12]:
# 2. Normalize and Deduplicate
# We lowercase and strip whitespace to identify near-duplicates
train_df['norm_text_for_dedup'] = train_df['cleaned_text'].str.lower().str.strip()

before_dedup = len(train_df)
train_df = train_df.drop_duplicates(subset=['norm_text_for_dedup'])
after_dedup = len(train_df)

print(f"Removed {before_dedup - after_dedup} duplicate reviews from the training set.")
print(f"New training set size: {after_dedup}")

Removed 0 duplicate reviews from the training set.
New training set size: 24903


In [ ]:
# 3. Analyze Explicit Ratings (Label Leakage)
train_df['has_explicit_rating'] = train_df['cleaned_text'].apply(find_explicit_ratings)
leakage_rate = train_df['has_explicit_rating'].mean() * 100

print(f"Percentage of training reviews containing explicit ratings (e.g., '8/10'): {leakage_rate:.2f}%")
display(train_df[train_df['has_explicit_rating']][['text', 'label']].head(3))

In [ ]:
# 4. Length Analysis (Word Count distribution)
train_df['word_count'] = train_df['cleaned_text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 5))
sns.histplot(train_df['word_count'], bins=50, kde=True, color='skyblue')
plt.axvline(230, color='red', linestyle='--', label='Average (~230 words)')
plt.axvline(512, color='purple', linestyle='--', label='512 Word-Pieces Boundary (approx)')
plt.title('Distribution of Review Word Counts')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.legend()
plt.show()

print(f"Mean word count: {train_df['word_count'].mean():.1f}")
print(f"Median word count: {train_df['word_count'].median():.1f}")
print(f"Percentage of reviews exceeding 512 words: {(train_df['word_count'] > 512).mean() * 100:.2f}%")

In [ ]:
# 5. Create Stratified Dev (Validation) Split of exactly 5,000 reviews
# We use stratify=y to maintain a balanced positive/negative ratio
dev_size = 5000

dev_df, train_split_df = train_test_split(
    train_df,
    test_size=(len(train_df) - dev_size),
    stratify=train_df['label'],
    random_state=42
)

print(f"Final train split size: {len(train_split_df)} (Pos: {(train_split_df['label'] == 1).sum()}, Neg: {(train_split_df['label'] == 0).sum()})")
print(f"Final dev split size: {len(dev_df)} (Pos: {(dev_df['label'] == 1).sum()}, Neg: {(dev_df['label'] == 0).sum()})")